# Klasifikasi KNN Manual (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini mengimplementasikan **K-Nearest Neighbors (KNN)** dari nol, tanpa
scikit-learn. Model dan metriknya dihitung sendiri. Datanya dibaca dari berkas bersih
hasil notebook `02_bersihkan_data`, jadi tidak ada library machine learning yang dipakai.

Tujuannya bukan mengejar skor tertinggi, melainkan melihat cara kerja algoritma dari
dalam: menghitung jarak, mengambil tetangga terdekat, lalu voting.

Dataset yang dipakai: Stroke Prediction versi bersih dari notebook `02_bersihkan_data`
(5.109 baris). Hanya kolom numerik yang dipakai: age, hypertension, heart_disease,
avg_glucose_level, bmi. Target: stroke.

Pasangan notebook ini untuk Naive Bayes ada di `03b_naive_bayes_manual.ipynb`.
Versi scikit-learn menyusul di `04_model_library.ipynb`.


## 1. Muat Dataset

Data diambil langsung dari **berkas dataset bersih** yang dihasilkan notebook
`02_bersihkan_data`, yaitu `artifacts/stroke_bersih.csv` (5.109 baris, `bmi` sudah diisi
prediksi Ridge, `gender` "Other" sudah dibuang).

Karena pembersihannya sudah selesai di notebook `02`, notebook ini bisa fokus penuh ke
algoritmanya. Kolom `gender` tetap dimuat karena dibutuhkan skor medis di bagian 5.

In [ ]:
import pandas as pd
import numpy as np
import math
import random

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

## 2. Train-Test Split Manual

Split 80:20 tanpa library. Acak indeks dulu, lalu potong. Karena datanya tidak
seimbang (hanya 4,87% stroke), proporsi kelas di train dan test dicetak supaya
kelihatan apakah kedua bagian kebagian kasus stroke.

In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

## 3. Fungsi Evaluasi Manual

Semua metrik dihitung dari confusion matrix tanpa library.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 4. KNN Manual

Hitung jarak Euclidean ke semua data train, ambil K tetangga terdekat, lalu voting
mayoritas.

In [ ]:
def euclidean_distance(a, b):
    """Jarak Euclidean antara dua vektor."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def knn_predict(X_train, y_train, X_test, k=5):
    """KNN: prediksi kelas untuk tiap titik uji."""
    y_pred = []

    for titik_uji in X_test:
        jarak = []
        for i, titik_latih in enumerate(X_train):
            d = euclidean_distance(titik_uji, titik_latih)
            jarak.append((d, y_train[i]))

        jarak.sort(key=lambda x: x[0])
        tetangga = jarak[:k]

        voting = 0
        for _, label in tetangga:
            voting += label

        pred = 1 if voting > k / 2 else 0
        y_pred.append(pred)

    return np.array(y_pred)


print("Menjalankan KNN (k=5)...")
y_pred_knn = knn_predict(X_train, y_train, X_test, k=5)
print("Selesai.")

In [ ]:
metrik_knn = hitung_metrik(y_test, y_pred_knn)

print("=" * 50)
print("HASIL KNN (k=5)")
print("=" * 50)
print(f"  Akurasi:  {metrik_knn['akurasi']}")
print(f"  Presisi:  {metrik_knn['presisi']}")
print(f"  Recall:   {metrik_knn['recall']}")
print(f"  F1-score: {metrik_knn['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_knn['tp']}  FP={metrik_knn['fp']}")
print(f"  FN={metrik_knn['fn']}  TN={metrik_knn['tn']}")

### 4a. Coba Beberapa Nilai K

Cari nilai K yang paling optimal dengan mencoba beberapa angka. K kecil peka derau,
K besar cenderung menebak kelas mayoritas.

In [ ]:
print(f"{'K':>3} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 40)

for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    pred = knn_predict(X_train, y_train, X_test, k=k)
    m = hitung_metrik(y_test, pred)
    print(f"{k:>3} {m['akurasi']:>8} {m['presisi']:>8} {m['recall']:>8} {m['f1']:>8}")

## 5. Pembanding: Skor Risiko Stroke Klinis (CHA2DS2-VASc)

Sebelum ada machine learning, dunia medis sudah punya cara menghitung risiko stroke.
Salah satu yang paling dikenal adalah **CHA2DS2-VASc**, skor yang dipakai untuk
memperkirakan risiko stroke pada pasien fibrilasi atrium.

Skornya hanyalah penjumlahan poin:

$$\text{Skor} = C + H + A_2 + D + S_2 + V + A + Sc$$

dengan tiap komponen bernilai:

| Lambang | Arti | Poin |
|---|---|---|
| $C$ | Congestive heart failure (gagal jantung) | 1 |
| $H$ | Hypertension (hipertensi) | 1 |
| $A_2$ | Age $\geq$ 75 tahun | 2 |
| $D$ | Diabetes mellitus | 1 |
| $S_2$ | Stroke atau TIA sebelumnya | 2 |
| $V$ | Vascular disease (penyakit vaskular) | 1 |
| $A$ | Age 65 sampai 74 tahun | 1 |
| $Sc$ | Sex category (perempuan) | 1 |

### Adaptasi ke dataset kami

Dua komponen tidak bisa dipakai, dan ini harus disebut apa adanya:

- $S_2$ (riwayat stroke) justru **label yang mau diprediksi**, jadi tidak boleh jadi fitur.
- $V$ (penyakit vaskular) tidak ada di dataset.

Sisanya dipakai apa adanya, dengan dua proksi yang juga harus disebut:

- $C$ memakai kolom `heart_disease`
- $D$ memakai `avg_glucose_level >= 126` mg/dL, ambang diabetes menurut ADA

Karena itu skor maksimum di sini **6**, bukan 9.

Skor ini tidak dilatih dan tidak belajar dari data, ia aturan tetap. Justru itu gunanya
di sini: jadi pembanding "cara dokter" terhadap "cara machine learning".

In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

## 6. Kesimpulan

KNN manual berhasil dijalankan dan dievaluasi, sepenuhnya tanpa scikit-learn. Hasilnya
justru memperlihatkan kelemahan KNN pada data ini dengan sangat jelas.

| K | Akurasi | Presisi | Recall | F1 |
|---|---|---|---|---|
| 1 | 0,9197 | 0,2308 | 0,1475 | 0,1800 |
| 3 | 0,9285 | 0,1250 | 0,0328 | 0,0519 |
| 5 | 0,9285 | 0,0000 | 0,0000 | 0,0000 |
| 21 | 0,9403 | 0,0000 | 0,0000 | 0,0000 |

Pada k=5 dan seterusnya, dari 61 pasien stroke di data uji tidak satu pun terdeteksi
(TP=0, FN=61). Akurasi 0,93 itu menyesatkan: model hanya menebak "tidak stroke".

### Dibandingkan aturan medis

Skor CHA2DS2-VASc di bagian 5 tidak dilatih sama sekali, hanya penjumlahan poin, tetapi
hasilnya jauh lebih berguna:

| Cara | Recall | Presisi | F1 |
|---|---|---|---|
| Skor medis, ambang skor >= 2 | 0,6885 | 0,1368 | 0,2283 |
| Skor medis, ambang skor >= 3 | 0,5082 | 0,2109 | 0,2981 |
| KNN terbaik, yaitu k=1 | 0,1475 | 0,2308 | 0,1800 |
| KNN k=5 | 0,0000 | 0,0000 | 0,0000 |

Pada data ini **aturan sederhana dari buku kedokteran mengalahkan KNN**. Penyebabnya
sudah terlihat di atas: KNN menghitung jarak pada fitur yang belum diskalakan, sehingga
keputusannya kacau.

Beberapa catatan penting:

1. **Skala fitur sangat berpengaruh.** KNN menghitung jarak, jadi `avg_glucose_level`
   (rentang puluhan sampai ratusan) mendominasi `hypertension` yang hanya 0/1. Jarak
   antar pasien nyaris ditentukan glukosa saja. Versi scikit-learn nanti perlu scaling,
   dan itu akan mengubah hasil.
2. **Recall kelas stroke hampir nol**, bukan karena datanya tidak punya sinyal,
   melainkan karena jarak yang timpang plus kelas yang sangat tidak seimbang membuat
   voting didominasi tetangga "tidak stroke".
3. **K kecil menyelamatkan sedikit recall** (k=1 mendeteksi 9 pasien, k=3 hanya 2),
   tetapi F1 terbaiknya pun cuma 0,18 pada k=1.
4. **Aturan medis jadi pembanding yang jujur.** Karena skor CHA2DS2-VASc tidak belajar
   dari data, ia tidak bisa dituduh menghafal. Kalau model tidak mengalahkannya, berarti
   modelnya belum layak dipakai.

Langkah berikutnya: implementasi manual yang sama untuk Naive Bayes di
`03b_naive_bayes_manual.ipynb`, lalu versi scikit-learn di `04_model_library.ipynb`.